# 알약 검수 파이프라인 서버 (Colab)

**사전 준비 (Google Drive에 업로드):**
```
MyDrive/pill_project/
  inspection_pipeline/   ← pipeline.py, server.py 등 파이프라인 파일들
  model/
    rtmdet_single_pill_aug_300.py
    best_coco_bbox_mAP_50_epoch_273.pth
```

**런타임 → 런타임 유형 변경 → GPU (T4) 선택 필수**

In [ ]:
# [1] GPU 확인
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음 - 런타임을 GPU로 변경하세요')

In [ ]:
# [2] 패키지 설치 (최초 1회, 약 5~10분 소요)
!pip install -q -U openmim
!mim install -q mmengine
!mim install -q 'mmcv>=2.0.0'
!mim install -q mmdet
!pip install -q fastapi uvicorn pyngrok python-multipart easyocr scipy

In [ ]:
# [3] Google Drive 마운트 및 파일 복사
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
src = '/content/drive/MyDrive/pill_project'
dst = '/content/pill_project'
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst)
print('복사 완료:')
!ls /content/pill_project/inspection_pipeline/

In [ ]:
# [4] 모델 경로 환경변수 설정
import os
os.environ['PILL_CONFIG']     = '/content/pill_project/model/rtmdet_single_pill_aug_300.py'
os.environ['PILL_CHECKPOINT'] = '/content/pill_project/model/best_coco_bbox_mAP_50_epoch_273.pth'
print('설정 완료')

In [ ]:
# [5] 서버 실행 + ngrok 터널
# ngrok.com 에서 무료 가입 후 토큰 발급: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = 'YOUR_NGROK_TOKEN_HERE'  # ← 여기에 토큰 입력

import subprocess, threading, time
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_TOKEN)

def run_server():
    subprocess.run(
        ['uvicorn', 'server:app', '--host', '0.0.0.0', '--port', '8000'],
        cwd='/content/pill_project/inspection_pipeline'
    )

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(4)

tunnel = ngrok.connect(8000)
SERVER_URL = tunnel.public_url
print(f'\n서버 실행 중!')
print(f'Android 앱에 입력할 주소: {SERVER_URL}')
print(f'분석 엔드포인트:          {SERVER_URL}/analyze')